# SHAP attribution — humans-only analysis

Compares token-level SHAP attributions for **human_high vs human_low** essays
per trait, using the trained RoBERTa probe as the model being explained.

Also runs the full humans-only linguistic analysis (LIWC, TF-IDF, keyword
frequency) and generates the two summary plots.

**Runtime**: ~20 min SHAP + ~3 min everything else on a T4 GPU.

**Before running**: merge the `analyze` branch to `main`, or uncomment
the `git checkout` line in Cell 1.

In [ ]:
# 1 — GPU check, clone, install
!nvidia-smi -L
!git clone https://github.com/avukovic11/evaluating-personality-expression-in-llms.git
%cd evaluating-personality-expression-in-llms
# uncomment if analyze branch not yet merged to main:
!git checkout analyze
!pip install -q -r requirements.txt
%cd code

GPU 0: Tesla T4 (UUID: GPU-8ff45910-7ff5-6da9-cdaa-a30ce3b37251)
Cloning into 'evaluating-personality-expression-in-llms'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (279/279), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 279 (delta 111), reused 236 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (279/279), 3.59 MiB | 6.36 MiB/s, done.
Resolving deltas: 100% (111/111), done.
/content/evaluating-personality-expression-in-llms/code/evaluating-personality-expression-in-llms
Branch 'analyze' set up to track remote branch 'analyze' from 'origin'.
Switched to a new branch 'analyze'
/content/evaluating-personality-expression-in-llms/code/evaluating-personality-expression-in-llms/code




re-train from scratch (~25 min). Use Cell 2b. The committed
splits guarantee an identical checkpoint.

In [ ]:
# Re-train from scratch
!python -m src.classifier --train --seeds 42

device : cuda
model  : roberta-base
seeds  : [42]
epochs : 8  batch=16  lr=2e-05
config.json: 100% 481/481 [00:00<00:00, 1.94MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 118kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 8.88MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 2.85MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 9.51MB/s]
Map: 100% 1973/1973 [00:04<00:00, 431.96 examples/s]
Map: 100% 246/246 [00:00<00:00, 351.87 examples/s]
Map: 100% 248/248 [00:00<00:00, 502.20 examples/s]

=== seed 42 ===
model.safetensors: 100% 499M/499M [00:02<00:00, 206MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 1025.44it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPEC

In [ ]:
# 3 — Verify checkpoint before running SHAP
from pathlib import Path
ckpt = Path("datasets/checkpoints/roberta-base_seed42")
contents = sorted(ckpt.iterdir()) if ckpt.exists() else []
if not contents:
    print("MISSING — run cell 2a or 2b first")
else:
    print(f"OK — {len(contents)} files in {ckpt}:")
    for f in contents:
        print(f"  {f.name:<40} {f.stat().st_size / 1e6:>7.1f} MB")

OK — 6 files in datasets/checkpoints/roberta-base_seed42:
  checkpoint-496                               0.0 MB
  config.json                                  0.0 MB
  model.safetensors                          498.6 MB
  tokenizer.json                               3.6 MB
  tokenizer_config.json                        0.0 MB
  training_args.bin                            0.0 MB


## Smoke test

Run SHAP on 5 essays per condition (~2 min). Confirms the checkpoint loaded
correctly and the output CSV schema looks right before committing to the
full 30-essay run.

In [ ]:
# 4 — Smoke test: SHAP only, 5 essays per condition
!python -m src.analyze --humans-only --shap --n-shap 5 \
    --skip-liwc --skip-tfidf --skip-keyword-freq

output : /content/evaluating-personality-expression-in-llms/code/evaluating-personality-expression-in-llms/code/datasets/results/llm-alignment/analysis/roberta-base/humans_only

=== Humans-only analysis (ground-truth label split) ===

=== SHAP token attribution — humans only (n_shap=5 per condition) ===
    Slow: ~20–60 s per essay on CPU, 3–8 s on GPU.
Loading weights: 100% 201/201 [00:00<00:00, 907.65it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
  cEXT / human_high: SHAP on 5 essays
Token indices sequence length is longer than the specified maximum sequence length for this model (862 > 512). Running this sequence through the model will result in indexing errors
PartitionExplainer explainer:  20% 1/5 [00:00<?, ?it/s]
  0% 0/498 [00:00<?, ?it/s]
 28% 140/498 [00:00<00:00, 579.93it/s]
 40% 200/498 [00:02<00:04, 61.04it/s] 
 46% 230/498 [00:03<00:05, 46.49it/s]
 50% 248/498 [00:04<00:06, 40.95it/s]
 52% 260/498 [00:05<00:06, 37.78it/s]
 55% 272/498 [00:05<00:06,

In [ ]:
# 4b — Inspect smoke-test output (one trait as a sanity check)
import pandas as pd
from pathlib import Path

out = Path("datasets/results/llm-alignment/analysis/roberta-base/humans_only")
df = pd.read_csv(out / "shap_cEXT.csv")
print(df.columns.tolist())
print(df.head(10).to_string(index=False))

['trait', 'condition', 'token', 'count', 'mean_abs_shap', 'mean_signed_shap']
trait  condition   token  count  mean_abs_shap  mean_signed_shap
 cEXT human_high  things      4       0.000832          0.000519
 cEXT human_high    nice      3       0.000798          0.000721
 cEXT human_high    into      3       0.000758          0.000758
 cEXT human_high  school      3       0.000757          0.000757
 cEXT human_high friends      4       0.000712          0.000712
 cEXT human_high   where      4       0.000707          0.000707
 cEXT human_high     are      5       0.000700          0.000697
 cEXT human_high  people      3       0.000696          0.000696
 cEXT human_high     now      3       0.000687          0.000687
 cEXT human_high weekend      3       0.000667          0.000667


## Full run

All analyses (LIWC + TF-IDF + keyword frequency + SHAP) plus the two
summary plots. Takes ~25 min total — SHAP dominates.

If the session dies mid-run, re-run this cell: the JSONL analyses
will redo instantly and SHAP will pick up where it left off (each
trait's CSV is written immediately after that trait finishes).

In [ ]:
# 5 — Full run
!python -m src.analyze --humans-only --shap --plot

In [ ]:
# 6 — Quick summary: top SHAP tokens per trait
import pandas as pd
from pathlib import Path

out = Path("datasets/results/llm-alignment/analysis/roberta-base/humans_only")
traits = ["cEXT", "cNEU", "cAGR", "cCON", "cOPN"]
names  = {"cEXT": "Extraversion", "cNEU": "Neuroticism", "cAGR": "Agreeableness",
          "cCON": "Conscientiousness", "cOPN": "Openness"}

for trait in traits:
    p = out / f"shap_{trait}.csv"
    if not p.exists():
        print(f"{trait}: not found"); continue
    df = pd.read_csv(p)
    print(f"\n{names[trait]}")
    for cond in ("human_high", "human_low"):
        top = df[df["condition"] == cond].head(5)
        tokens = ", ".join(
            f"{r['token']} ({r['mean_signed_shap']:+.4f})"
            for _, r in top.iterrows()
        )
        print(f"  {cond}: {tokens}")

## Download and (optionally) commit results

In [ ]:
# 7 — Download all humans_only results as a zip
import shutil
from google.colab import files

shutil.make_archive(
    "/content/humans_only_results", "zip",
    "datasets/results/llm-alignment/analysis/roberta-base/humans_only",
)
files.download("/content/humans_only_results.zip")